<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/01_opendss_workflow_with_py_dss_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — OpenDSS workflow with py-dss-interface

The goal of this notebook is simple: take the workflow we usually run from a `Run_ckt5.dss` file inside OpenDSS and reproduce it from Python.

The key idea is that **OpenDSS is controlled through text commands**. The `compile`, `New`, `Edit`, `set`, `solve`, `show`, and `export` commands you type in the OpenDSS standalone are exactly the same strings you can send from Python through `dss.text(...)`. That is what `py-dss-interface` does — it gives Python a direct line to OpenDSS so we can drive the simulation from a script instead of typing in the GUI.

In this notebook we use **only** `py-dss-interface`. That is intentional: I want the bridge between the run file and Python to feel obvious. In Notebook 4 we will see how `py-dss-toolkit` makes a lot of this work shorter.

## Setup

Install `py-dss-interface` and clone the repository so we have access to the `ckt5` feeder.

In [ ]:
!pip install py-dss-interface
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

## Two files, two responsibilities

When we work with OpenDSS we usually deal with two files:

- **Master file** — defines the feeder model: circuit, transformers, lines, loads, regulators, generators. There is no simulation here, just the model.
- **Run file** — loads the master file, sets simulation options, solves, and asks for results.

Think of it this way: the master file is *what the feeder is*, and the run file is *what we want to do with it*.

What we will do in this notebook is take the run file and replace it with Python code that sends the same commands.

In [ ]:
import pathlib
import py_dss_interface

# Build a portable path to the master file based on the cloned repo.
start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "EPRITestCircuits" / "ckt5" / "Master_ckt5.dss"

print(f"Master file: {dss_file}")

## Create the OpenDSS object

`py_dss_interface.DSS()` starts an OpenDSS engine inside Python. From now on, every command we send through `dss.text(...)` is the same string we would type in the OpenDSS standalone.

In [ ]:
dss = py_dss_interface.DSS()
print(f"OpenDSS started: {dss.started}")
print(f"OpenDSS version: {dss.dssinterface.version}")

## Compile the master file

`compile` loads the master file and runs every command it contains. After this step the feeder model lives in OpenDSS memory, but we have not solved anything yet.

**Tip:** I recommend always passing the **full path** to `compile`. OpenDSS keeps a current data path, and using the full path means our script does not depend on which folder OpenDSS is currently pointing to.

In [ ]:
dss.text(f"compile [{dss_file}]")

# Confirm the OpenDSS data path now points to ckt5.
print(f"OpenDSS data path: {dss.dssinterface.datapath}")

## Send the same commands as the run file

Below is what `Run_ckt5.dss` does for a snapshot solve, translated one line at a time. Each `dss.text(...)` call is exactly the string we would type in OpenDSS.

In [ ]:
# Make the energymeter local so it does not propagate work over the whole circuit.
dss.text("Energymeter.sub.local=yes losses=no")

# Add a few monitors at the substation transformer and feederhead line.
dss.text("New monitor.ckt5_totalizedmonitor Transformer.MDV_SUB_1 term=2 mode=1")
dss.text("New monitor.ckt5_mon element=Line.MDV201_connector terminal=1 mode=0 Residual=yes")
dss.text("New monitor.ckt5_mon_p element=Line.MDV201_connector terminal=1 mode=1 Ppolar=No")

## Set simulation parameters and solve

OpenDSS uses snapshot mode by default, but it is good practice to set it explicitly so the script is self-documenting.

In [ ]:
dss.text("set mode=snapshot")
dss.text("solve")

print(f"Converged: {dss.solution.converged}")

## Get basic results

Now that the power flow is solved, we read results directly from `py-dss-interface`. Two of the most useful access points are:

- `dss.circuit.total_power` — total active and reactive power at the feederhead in kW and kvar.
- `dss.circuit.losses` — total active and reactive losses in W and var.
- `dss.circuit.buses_vmag_pu` and `dss.circuit.nodes_names` — bus voltages in per unit and the matching node names.

In [ ]:
p_kw, q_kvar = dss.circuit.total_power
p_loss_w, q_loss_var = dss.circuit.losses

print(f"Feederhead P: {-p_kw:>12,.1f} kW")
print(f"Feederhead Q: {-q_kvar:>12,.1f} kvar")
print(f"Total losses (active):   {p_loss_w/1000:>10,.2f} kW")
print(f"Total losses (reactive): {q_loss_var/1000:>10,.2f} kvar")

In [ ]:
voltages_pu = dss.circuit.buses_vmag_pu
node_names = dss.circuit.nodes_names

print(f"Number of nodes: {len(voltages_pu)}")
print(f"Min voltage: {min(voltages_pu):.4f} pu")
print(f"Max voltage: {max(voltages_pu):.4f} pu")

## The text method also returns strings

Some OpenDSS commands return useful information when we send them through `dss.text(...)`. `export voltages`, for example, writes a CSV file and returns the path to it. That can be handy when we want to capture an OpenDSS report without writing the file logic ourselves.

In [ ]:
export_path = dss.text("export voltages")
print(f"Voltages exported to: {export_path}")

## Modify the model from Python

The same `dss.text(...)` channel is also how we change the model. As an example, we can scale all loads up by 5% with one command and re-solve.

In [ ]:
dss.text("set loadmult=1.05")
dss.text("solve")

p_kw_new, q_kvar_new = dss.circuit.total_power
print(f"Feederhead P at 105% load: {-p_kw_new:,.1f} kW")
print(f"Feederhead P delta vs base: {(-p_kw_new) - (-p_kw):,.1f} kW")

## Key takeaways

- The **master file** is the feeder model. The **run file** is the simulation script. Keep them separated.
- Anything you can do in OpenDSS, you can do from Python via `dss.text(...)`. That is the bridge.
- After `solve`, results are already in OpenDSS memory and you read them through `py-dss-interface` properties — no need to export and re-read CSV files unless you want to.
- Use the **full path** in `compile` so your script does not depend on the OpenDSS data path.

OpenDSS works fine on its own. What Python adds — and what we will see in the next notebooks — is the ability to *automate* the run file. Loops, conditionals, comparisons, and visualizations all become natural once the simulation lives inside a Python script.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)